# FutureWork AI — Demand-Aware Career Navigator
**Datathon extension · IBM LinuxONE · O*NET skills + optional verified Jobs and Skills Australia demand evidence**

**Important:** Our O*NET data are **US occupation skill profiles**. Australian labour-market observations have **ANZSCO/OSCA** codes, which must be linked with an explicit, reviewed crosswalk. This notebook does not assume that matching titles proves equivalence. **Unmapped occupations remain UNKNOWN, never LOW demand.**

Data: [JSA employment projections](https://www.jobsandskills.gov.au/data/employment-projections) and [JSA occupation shortage list](https://www.jobsandskills.gov.au/data/occupation-shortage). Keep copies/source URLs and publication dates with your submission.

**Safety:** Demand indicators are not vacancies, future guarantees, or hiring probabilities. We display every score component.

## 1 — Load the prepared Phase 1 data
Keep this notebook next to Phases 1–3, and check `processed/occupation_skill_matrix.csv` plus `processed/occupation_lookup.csv` exist.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT=Path.cwd(); PROCESSED=ROOT/'processed'; DATA=ROOT/'data'; RESULTS=ROOT/'results'
DATA.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)
for file in ('occupation_skill_matrix.csv','occupation_lookup.csv'):
    if not (PROCESSED/file).exists(): raise FileNotFoundError(f'Missing {PROCESSED/file}. Run Phase 1 first.')
raw=pd.read_csv(PROCESSED/'occupation_skill_matrix.csv',dtype={'occupation_code':str})
lookup=pd.read_csv(PROCESSED/'occupation_lookup.csv',dtype={'occupation_code':str}).drop_duplicates('occupation_code').set_index('occupation_code')
X=raw.set_index('occupation_code').apply(pd.to_numeric,errors='coerce')
X=X.loc[X.index.intersection(lookup.index)]
coverage=X.notna().mean(axis=1); eligible=coverage[coverage>=0.60].index
filled=X.loc[eligible].fillna(X.loc[eligible].median()).fillna(1)
SIM=pd.DataFrame(cosine_similarity(np.maximum(filled.to_numpy()-1,0)),index=eligible,columns=eligible)
print(f'Loaded {len(eligible)} eligible occupations / {len(X.columns)} skills')

## 2 — Data contract: reviewed crosswalk and market signals

Two simple CSVs connect Australian demand evidence to the existing system. This is deliberately an **auditable manual mapping process** for the Datathon, rather than an unreliable fuzzy join:

`data/verified_mapping.csv`: `occupation_code,au_code,au_classification,reviewed,source_note`

`data/au_market_signals.csv`: `au_code,au_classification,growth_pct_10yr,shortage_status,source_note`

- `occupation_code`: exact O*NET code, e.g. `15-1252.00`.
- `au_code`: exact **6-digit** Australian occupation code from your official source.
- `au_classification`: `ANZSCO` or `OSCA`, consistent across the mapping and market file.
- `reviewed`: `yes` only if a human checked equivalence; otherwise the row is ignored.
- `growth_pct_10yr`: published 10-year projected percentage growth **or blank** (not made-up forecasts).
- `shortage_status`: `shortage`, `no_shortage` or blank; a shortage observation is **not** a job-growth forecast.
- `source_note`: source, release year, and optionally a link for audit.

Download XLSX files from the official links above; inspect the sheets and *manually copy a small subset of validated values* into the simple CSV format. Do not guess occupation codes. The notebook creates blank templates for your convenience.

In [ ]:
mapfile=DATA/'verified_mapping.csv'; marketfile=DATA/'au_market_signals.csv'
if not mapfile.exists():
    pd.DataFrame(columns=['occupation_code','au_code','au_classification','reviewed','source_note']).to_csv(mapfile,index=False)
if not marketfile.exists():
    pd.DataFrame(columns=['au_code','au_classification','growth_pct_10yr','shortage_status','source_note']).to_csv(marketfile,index=False)
print('Mapping template:',mapfile,'| Labour-market template:',marketfile)
print('Fill with VERIFIED data if you want real demand-aware rankings. Never treat blank values as low demand.')

## 3 — Read and validate market signals
When no verified data are supplied, results remain **skills-only** and demand is labelled **unknown**. We intentionally do *not* insert fictional growth/shortage figures.

In [ ]:
M=pd.read_csv(mapfile,dtype=str).fillna(''); A=pd.read_csv(marketfile,dtype=str).fillna('')
mapcols={'occupation_code','au_code','au_classification','reviewed','source_note'}
marketcols={'au_code','au_classification','growth_pct_10yr','shortage_status','source_note'}
assert mapcols.issubset(M.columns) and marketcols.issubset(A.columns), 'Required columns changed.'
for frame in (M,A):
    frame['au_code']=frame.au_code.str.strip(); frame['au_classification']=frame.au_classification.str.strip().str.upper()
    valid_codes=(frame.au_code=='') | frame.au_code.str.fullmatch(r'\d{6}')
    assert valid_codes.all(), 'Australian occupation codes must have 6 digits.'
    assert frame.loc[frame.au_code!='','au_classification'].isin(['ANZSCO','OSCA']).all()
M['occupation_code']=M.occupation_code.str.strip()
M=M[(M.reviewed.str.strip().str.lower()=='yes') & (M.occupation_code.isin(eligible)) & (M.au_code!='')].copy()
A=A[A.au_code!=''].copy()
assert not M.occupation_code.duplicated().any(), 'Resolve ambiguous multiple mappings by human review.'
assert not A.duplicated(['au_code','au_classification']).any(), 'Duplicate AU records.'
assert A.shortage_status.str.lower().isin(['','shortage','no_shortage']).all(), 'Unexpected shortage label.'
A['growth_pct_10yr']=pd.to_numeric(A.growth_pct_10yr.replace('',np.nan),errors='raise')
assert np.isfinite(A.growth_pct_10yr.dropna()).all()
joined=M.merge(A,on=['au_code','au_classification'],how='left',suffixes=('_map','_market'),validate='one_to_one')
market=joined.set_index('occupation_code') if not joined.empty else pd.DataFrame(index=pd.Index([],name='occupation_code'))
print(f'Verified occupation mappings: {len(M)}; with Australian market rows: {len(joined[joined.source_note_market.notna()]) if len(joined) else 0}')
print('No Australian demand evidence? That is okay: the skills-only model continues without inventing facts.')

## 4 — Ranking algorithm and missing-evidence policy
**Skills-only** ranking is always available. **Opportunity-aware** ranking activates *only* when there is verified demand evidence, and scores only candidates with evidence. Others are explicitly labelled `UNKNOWN`, not penalised as low demand.

Illustrative, tunable policy: 70% skill similarity + 30% growth percentile **within the mapped comparison set**. Shortage status is displayed separately and is **not** conflated with growth. We require at least three occupations with growth data before computing a demand-adjusted rank. This is not a validated employment prediction.

In [ ]:
def demand_recommend(source_code,top_n=5):
    source_code=str(source_code)
    if source_code not in SIM.index: raise ValueError('Not in eligible O*NET occupations')
    scores=SIM.loc[source_code].drop(source_code).rename('similarity').reset_index().rename(columns={'index':'occupation_code'})
    scores['occupation_title']=scores.occupation_code.map(lookup.occupation_title)
    scores['growth_pct_10yr']=np.nan; scores['shortage_status']='unknown'
    scores['au_code']=''; scores['au_classification']=''
    if not market.empty:
        for col in ('growth_pct_10yr','shortage_status','au_code','au_classification'):
            if col in market.columns:
                vals=scores.occupation_code.map(market[col])
                if col=='growth_pct_10yr': scores[col]=pd.to_numeric(vals,errors='coerce')
                else: scores[col]=vals.fillna('unknown' if col=='shortage_status' else '')
    scores['evidence_status']=np.where(scores.growth_pct_10yr.notna(),'verified_growth','no_verified_growth')
    skilltop=scores.sort_values('similarity',ascending=False).head(top_n).copy()
    valid=scores[scores.growth_pct_10yr.notna()].copy()
    if len(valid)>=3 and valid.growth_pct_10yr.nunique()>1:
        valid['growth_percentile']=valid.growth_pct_10yr.rank(pct=True)
        valid['combined_score']=0.70*valid.similarity+0.30*valid.growth_percentile
        opportunity=valid.sort_values('combined_score',ascending=False).head(top_n).copy()
        status=f'Opportunity-aware ranking based on {len(valid)} mapped, evidence-bearing alternatives (not all 910).'
    else:
        opportunity=None
        status='Insufficient verified comparable growth data: opportunity-aware ranking DISABLED.'
    return skilltop,opportunity,status,scores

# Choose any supported occupation by title:
def search_jobs(text):
    return lookup.loc[eligible][lookup.loc[eligible,'occupation_title'].str.contains(text,case=False,na=False)].head(12)[['occupation_title']]
display(search_jobs('software'))

## 5 — Demonstrate with real O*NET skills; demand only if validated data exist

In [ ]:
example_matches=search_jobs('software developer')
source_code=(example_matches.index[0] if len(example_matches) else eligible[0])
print('Source:',lookup.loc[source_code,'occupation_title'],'|',source_code)
skill_top,opportunity,status,all_candidates=demand_recommend(source_code)
print('\nTop five SKILLS-ONLY recommendations:')
display(skill_top[['occupation_title','similarity','growth_pct_10yr','evidence_status']])
print('\n',status)
if opportunity is not None:
    display(opportunity[['occupation_title','similarity','growth_pct_10yr','shortage_status','combined_score']])
else:
    print('No demand-ranked recommendations displayed. Add verified Australian mappings/data first.')
fig,ax=plt.subplots(figsize=(9,4))
plot=skill_top.iloc[::-1]
ax.barh(plot.occupation_title,plot.similarity)
ax.set_xlim(0,1.02); ax.set_xlabel('O*NET skill-profile cosine similarity (not probability of employment)')
ax.set_title('Best occupational skill matches');plt.tight_layout();plt.show()

## 6 — Interactive source occupation and two ranking views (optional widgets)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    opts=sorted([(lookup.loc[c,'occupation_title']+' | '+c,c) for c in eligible],key=lambda p:p[0].lower())
    search=widgets.Text(placeholder='Search an occupation...',description='Search:')
    choose=widgets.Dropdown(options=opts,value=source_code,description='Occupation:',layout=widgets.Layout(width='75%'))
    output=widgets.Output(); button=widgets.Button(description='Compare rankings',button_style='success')
    def filteropts(change):
        term=change['new'].lower().strip()
        subset=[a for a in opts if term in a[0].lower()]
        if subset: choose.options=subset
    search.observe(filteropts,names='value')
    def run(_):
        with output:
            clear_output(wait=True)
            s,o,message,_=demand_recommend(choose.value)
            display(Markdown('### Skill-only matches'))
            display(s[['occupation_title','similarity','growth_pct_10yr','evidence_status']])
            display(Markdown('**'+message+'**'))
            if o is not None:
                display(Markdown('### Market-evidence-aware matches'))
                display(o[['occupation_title','similarity','growth_pct_10yr','combined_score']])
    button.on_click(run)
    display(widgets.VBox([search,choose,button,output]))
except ImportError:
    print('Widgets unavailable. Manual result above remains usable.')

## 7 — Tests, transparent evidence coverage, and exports
A very high similarity score is **not** model accuracy. Market-data coverage needs separate reporting.

In [ ]:
assert len(skill_top)==5 and source_code not in set(skill_top.occupation_code)
assert skill_top.similarity.between(0,1.000001).all()
assert skill_top.similarity.is_monotonic_decreasing
assert not skill_top.occupation_code.duplicated().any()
if opportunity is not None:
    assert opportunity.growth_pct_10yr.notna().all()
    assert opportunity.combined_score.is_monotonic_decreasing
skill_top.to_csv(RESULTS/'phase4_skill_only.csv',index=False)
if opportunity is not None: opportunity.to_csv(RESULTS/'phase4_demand_aware.csv',index=False)
coverage_total=int(all_candidates.growth_pct_10yr.notna().sum())
pd.DataFrame([{'eligible_alternative_occupations':len(all_candidates),'mapped_with_verified_growth':coverage_total,'market_data_coverage_pct':round(100*coverage_total/len(all_candidates),2),'demand_ranking_enabled':opportunity is not None}]).to_csv(RESULTS/'phase4_evidence_coverage.csv',index=False)
print('Finished. Australian market evidence available for',coverage_total,'of',len(all_candidates),'candidate occupations.')
print('Exports saved to:',RESULTS)
print('Never describe unverified demand as low demand or similarity as hiring probability.')

## Data entry and verification checklist
1. Open [JSA 2025–2035 Employment Projections](https://www.jobsandskills.gov.au/data/employment-projections) and download the official Excel workbook.
2. Open [JSA 2025 Occupation Shortage List](https://www.jobsandskills.gov.au/data/occupation-shortage) and download the workbook. Both official spreadsheets may contain different layouts/sheets. Do **not** feed them to this notebook without transforming them into the stated columns.
3. For a demo, choose **a small subset of occupations** (e.g., 5–20), verify each O*NET ↔ ANZSCO/OSCA mapping manually using occupational duties, and record exact codes/source in `verified_mapping.csv`.
4. Copy the verified 10-year percentage growth and shortage status into `au_market_signals.csv`; leave unsupported fields blank.
5. Re-run cells 3 onward. Show the coverage number as well as recommendations. If data are sparse, the notebook correctly declines to make demand-based recommendations.

**A data table with real Australian market evidence is more useful than fabricated scores for all 910 occupations.**

### Limitations
Skills importance is not an individual's proficiency. ANZSCO/OSCA equivalence can be imperfect; training/licensing and location are not incorporated. Employment projections are uncertain; shortages are point-in-time recruitment constraints, not guarantees. This prototype is a decision-support aid, not career advice.